# 6. TensorRT build and inference

Build engines on the target NVIDIA GPU. Install the TensorRT extras and CUDA compiler/toolchain first. The standard cached runtime is static batch 1. A list is not GPU batching. GPU/software-specific engines are generated locally and are not committed.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
RUN_BUILD = os.environ.get("RUN_TRT", "0") == "1"
RUN_TENSORRT = RUN_BUILD
if RUN_BUILD:
    run("build")
if RUN_TENSORRT:
    run("infer", "--backend", "tensorrt", "--image", str(IMAGE_PATH))
    run("evaluate", "--backend", "tensorrt", "--folder", str(config["dataset_root"] / "val"), "--output", str(config["output_dir"] / "evaluation_tensorrt"))
else:
    print("Set RUN_TRT=True only on a configured NVIDIA/CUDA system after ONNX export.")


In [ ]:
RUN_TRT_PARITY = RUN_TENSORRT and checkpoint.is_file()
if RUN_TRT_PARITY:
    from PIL import Image
    from waterec_ocr.waterec_models import TorchRecognizer
    from waterec_ocr.waterec_runtime import TensorRTRecognizer
    native = TorchRecognizer(config, checkpoint)
    engine = TensorRTRecognizer(deployment)
    images = [Image.open(p).convert("RGB") for p in examples]
    try:
        expected, actual = native.infer(images), engine.infer(images)
        print(list(zip(expected, actual)))
        assert expected == actual, "TensorRT changed predictions; do not accept this deployment yet"
    finally:
        engine.close()


Report actual native/engine prediction differences. A successful builder or identical dummy strings does not establish full-benchmark accuracy. The optional GPU preprocessing path is covered in notebook 8 and requires cc/nvcc.